In [149]:
import os
import copy
import numpy as np
import pandas as pd
import geopandas as gpd

from pathlib import Path

from src import utils, stats_utils
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_context("paper", font_scale=1.25)

from src.data.lfs import EuLfs
from src.data.framework import Esco, Classifications

# load paths
useful_paths = utils.UsefulPaths()

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Read data

##### Covariates
- NACE labels
- GBN shares based on unweighted classifications
- Earnings deciles data

NACE labels

In [150]:
# for NACE labels
classifications = Classifications()
covariates_by_nace = classifications.nace_1d

Country-specific earnings deciles

In [169]:
# earnings deciles
earnings_deciles = pd.read_csv(
    os.path.join(useful_paths.data_raw, "metadata", "earnings_per_deciles.csv"),
    #dtype={"decile": str}
)

GBN shares and categories based on unweighted classifications

In [152]:
# GBN shares
esco = Esco()
gbn_shares_no_wt = esco.read_gbn_classification(agg_to_isco_at_digit=3)

gbn_shares_no_wt = gbn_shares_no_wt.rename(
    columns={"preferredLabel_isco": "ISCO3D_label"}
)
gbn_shares_no_wt["NOBS"] = 1

# define the GBN category of an ISCO 3D group as the one with the highest fraction
# Note: brown category currently assigned based on SL
# Todo: could assign based on SLT as well (scenario-specific),
#  further degree of freedom
gbn_shares_no_wt["category_sl"] = gbn_shares_no_wt[
    ["share_green", "share_brown_sl", "share_neutral_sl"]
].idxmax(axis=1)

gbn_shares_no_wt["category_slt"] = gbn_shares_no_wt[
    ["share_green", "share_brown_slt", "share_neutral_slt"]
].idxmax(axis=1)

gbn_shares_no_wt = gbn_shares_no_wt.replace(
    to_replace={
        "category_sl": {
            "share_green": "green",
            "share_brown_sl": "brown",
            "share_neutral_sl": "neutral",
        },
        "category_slt": {
            "share_green": "green",
            "share_brown_slt": "brown",
            "share_neutral_slt": "neutral",
        }
    }
)

gbn_shares_no_wt

gbn_classification_short_list,isco_code,ISCO3D_label,share_green,share_brown_sl,share_neutral_sl,share_brown_slt,share_neutral_slt,NOBS,category_sl,category_slt
0,011,Commissioned armed forces officers,0.00,0.0,1.00,0.0,1.00,1,neutral,neutral
1,021,Non-commissioned armed forces officers,0.00,0.0,1.00,0.0,1.00,1,neutral,neutral
2,031,"Armed forces occupations, other ranks",0.00,0.0,1.00,0.0,1.00,1,neutral,neutral
3,111,Legislators and senior officials,0.00,0.0,1.00,0.0,1.00,1,neutral,neutral
4,112,Managing directors and chief executives,0.00,0.0,1.00,0.0,1.00,1,neutral,neutral
...,...,...,...,...,...,...,...,...,...,...
120,941,Food preparation assistants,0.00,0.0,1.00,0.0,1.00,1,neutral,neutral
121,951,Street and related service workers,0.00,0.0,1.00,0.0,1.00,1,neutral,neutral
122,952,Street vendors (excluding food),0.00,0.0,1.00,0.0,1.00,1,neutral,neutral
123,961,Refuse workers,0.75,0.0,0.25,0.0,0.25,1,green,green


##### LFS data

In [153]:
reprocess=False

config_file = "eu_lfs_config.yml"
config = utils.load_config(os.path.join(useful_paths.config_dir, config_file))
eulfs = EuLfs(config=config)

In [154]:
year = 2019

countries_isco3d = [
    "AT",
    "BE",
    "CH",
    "CY",
    "CZ",
    "DE",
    "DK",
    "EE",
    "ES",
    "FI",
    "FR",
    "GR",
    "HR",
    "HU",
    "IE",
    "IS",
    "IT",
    "LT",
    "LU",
    "LV",
    "NL",
    "NO",
    "PT",
    "RO",
    "SE",
    "SK",
    "UK",
]

if reprocess:
    eulfs.preprocess_files(
        years=[year],
        countries=countries_isco3d,
        save_file=True,
        optional_output_dir=config["paths"]["interim"],
        output_fname="eu_lfs_merged_{year}",
    )

In [155]:
df_all = eulfs.read_preprocessed_file(year=year)
df_all

,SEX,WSTATOR,NACE1D,ISCO3D,COUNTRYW,REGIONW,FTPT,REFYEAR,COUNTRY,REGION,...,HHTYPE,COEFF,AGE,ILOSTAT,EDUC4WN,HATLEV1D,MAINSTAT,INCDECIL,NUTS_ID,ISCO
0,1,1,C,132,AT,12,1,2019,AT,10,...,1,82.5525,52,1,0,M,1,NaN,AT12,132
1,1,1,G,311,AT,12,1,2019,AT,10,...,1,82.5525,17,1,1,L,1,NaN,AT12,311
2,2,1,G,411,AT,12,1,2019,AT,10,...,1,47.2325,42,1,0,M,1,NaN,AT12,411
3,2,1,Q,911,AT,12,1,2019,AT,10,...,1,52.9450,47,1,0,M,1,NaN,AT12,911
4,2,1,R,265,AT,12,1,2019,AT,10,...,1,75.8625,42,1,0,H,1,NaN,AT12,265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1417927,1,2,G,421,UK,M0,2,2019,UK,M0,...,1,1100.1400,17,1,1,M,NaN,NaN,UKM,421
1417928,1,1,O,211,UK,M0,1,2019,UK,M0,...,1,1790.6100,52,1,0,H,NaN,10,UKM,211
1417929,1,1,H,833,UK,M0,1,2019,UK,M0,...,1,1858.6800,57,1,0,M,NaN,NaN,UKM,833
1417930,2,1,Q,263,UK,M0,1,2019,UK,M0,...,1,1723.3000,57,1,0,H,NaN,06,UKM,263


#### Descriptive statistics

**Sample size by sector and country**
Sectoral focus on key sectors affected by green transition

In [156]:
sample_size_by_ind_and_country = (
    df_all.groupby(["NACE1D", "COUNTRY"]).count()["COEFF"].unstack()
)
sample_size_by_ind_and_country.to_csv(
    os.path.join(
        useful_paths.figure_dir, "03_eulfs", "eu_lfs_sample_size_by_ind_and_country.csv"
    )
)
sample_size_by_ind_and_country

COUNTRY,AT,BE,CH,CY,CZ,DE,DK,EE,ES,FI,...,LT,LU,LV,NL,NO,PT,RO,SE,SK,UK
NACE1D,,,,,,,,,,,,,,,,,,,,,
A,2992,239,966,466,519,3186,1055,533,1729,439,...,1742,40,325,720,311,4406,23356,1043,1048,463
B,138,12,21,33,104,448,82,110,71,35,...,54,1,16,35,346,188,782,124,130,138
C,13974,2254,4948,1250,4850,50290,5243,2613,4986,1460,...,4685,163,571,3789,1115,9052,19705,6710,8462,3642
D,602,120,233,97,186,2089,281,127,164,59,...,241,26,47,173,101,277,970,398,405,224
E,403,160,122,142,216,1656,272,65,284,45,...,443,13,25,178,67,558,1175,347,354,262
F,7212,1445,2191,1679,1198,17448,2822,1088,2379,848,...,2124,244,300,1783,1188,4054,6988,4537,2891,2786
G,12669,2505,4647,3171,1905,36325,7795,1904,5816,1226,...,4797,339,581,6059,1872,9120,14422,7104,4280,4818
H,4297,1040,1635,757,1104,13074,1964,997,1819,656,...,2154,202,308,1763,686,2775,5561,3094,2387,1903
I,5474,735,1406,1532,581,9243,1999,589,2886,383,...,782,172,118,1740,457,4938,2257,1933,1412,1955


**Number of unique occupation groups by sector and country**
Sectoral focus on key sectors affected by green transition

In [157]:
n_unique_occ_by_ind_and_country = (
    df_all.groupby(["NACE1D", "COUNTRY"])["ISCO3D"].nunique().unstack()
)
n_unique_occ_by_ind_and_country.to_csv(
    os.path.join(
        useful_paths.figure_dir, "03_eulfs", "n_unique_occ_by_ind_and_country.csv"
    )
)
n_unique_occ_by_ind_and_country

COUNTRY,AT,BE,CH,CY,CZ,DE,DK,EE,ES,FI,...,LT,LU,LV,NL,NO,PT,RO,SE,SK,UK
NACE1D,,,,,,,,,,,,,,,,,,,,,
A,53,28,70,22,50,79,55,42,59,34,...,62,7,47,75,42,60,62,51,55,54
B,26,9,13,11,26,59,25,24,18,17,...,16,1,9,23,42,26,40,20,22,38
C,107,95,120,63,79,113,94,78,88,76,...,94,53,71,117,80,95,94,84,105,98
D,48,40,48,18,44,85,42,35,39,22,...,38,8,18,48,29,38,46,41,40,46
E,43,41,37,21,43,82,40,18,43,15,...,50,8,13,57,24,54,50,38,44,47
F,85,67,82,39,48,96,61,48,60,54,...,64,39,38,88,57,60,60,70,61,81
G,112,89,117,72,73,113,91,64,88,76,...,95,59,64,122,73,93,81,95,88,93
H,86,64,94,40,56,103,70,59,58,51,...,65,41,54,92,48,72,58,66,65,73
I,88,46,83,46,31,89,60,40,61,26,...,49,19,25,69,35,66,51,44,42,55


##### Regional distribution of unique occupations
Needed for regional reskilling constraint.

In Germany, over 90% of all unique 3-digit occupations have COEFF > 0 in each NUTS2 region

In [187]:
df = df_all[df_all.COUNTRYW == "DE"]
n_isco_max = df.ISCO.unique().shape[0]
occ_number_by_nuts2_and_isco3 = df.groupby(["NUTS_ID", "ISCO3D"]).aggregate({"COEFF": np.sum})
occ_number_by_nuts2_and_isco3

n_unique_occ_by_nuts2 = occ_number_by_nuts2_and_isco3.reset_index().groupby("NUTS_ID").aggregate({"COEFF": np.count_nonzero})
n_unique_occ_by_nuts2 = n_unique_occ_by_nuts2.rename(columns={"COEFF": "n_unique_abs"})
n_unique_occ_by_nuts2["n_unique_rel"] = n_unique_occ_by_nuts2["n_unique_abs"] / n_isco_max
n_unique_occ_by_nuts2

,n_unique_abs,n_unique_rel
NUTS_ID,,
DE11,115,0.966387
DE12,114,0.957983
DE13,115,0.966387
DE14,113,0.949580
DE21,115,0.966387
DE22,109,0.915966
DE23,112,0.941176
DE24,108,0.907563
DE25,113,0.949580


##### Missing earnings information (INCDECIL) per country
Many countries totally lack any data on earnings

In [158]:
missing_lfs_data_incdecile = df_all.groupby("COUNTRYW")["INCDECIL"].apply(utils.perc_missing).reset_index().drop(labels="level_1", axis=1)
missing_lfs_data_incdecile = missing_lfs_data_incdecile.sort_values("missing_obs_rel", ascending=False)
missing_lfs_data_incdecile.to_csv(os.path.join(useful_paths.figure_dir, "03_eulfs", "eulfs_perc_missing_incdecile_per_country.csv"))
missing_lfs_data_incdecile

,COUNTRYW,missing_obs_abs,missing_obs_rel
0,AT,86056.0,1.000000
24,SE,67735.0,1.000000
21,NO,14150.0,1.000000
4,CZ,17143.0,1.000000
15,IS,9105.0,1.000000
8,ES,40695.0,1.000000
9,FI,8974.0,0.803402
14,IE,40136.0,0.635224
11,GR,33797.0,0.456180
26,UK,14117.0,0.368446


##### Imputation of INCDECIL variable
We therefore impute missing deciles by country-specific occupuation-industry medians

In [159]:
# calculate median income decile per country-occupation-industry group
df_all["INCDECIL"] = df_all["INCDECIL"].astype(float)
incdecil_by_cnt_isco_nace = df_all.groupby(["COUNTRYW", "ISCO3D", "NACE1D"]).aggregate({"INCDECIL": np.nanmedian}).reset_index()
incdecil_by_cnt_isco_nace = incdecil_by_cnt_isco_nace.rename(columns={"INCDECIL": "INCDECIL_median"})
incdecil_by_cnt_isco_nace["INCDECIL_median_floor"] = np.floor(incdecil_by_cnt_isco_nace['INCDECIL_median'])
incdecil_by_cnt_isco_nace["INCDECIL_median_ceil"] = np.ceil(incdecil_by_cnt_isco_nace['INCDECIL_median'])

# join to LFS data
df_all = pd.merge(df_all, incdecil_by_cnt_isco_nace, on=["COUNTRYW", "ISCO3D", "NACE1D"])

# impute with ceiled medians (more conservative estimate)
df_all["INCDECIL_imputed"] = df_all["INCDECIL"].fillna(df_all["INCDECIL_median_ceil"])

Save overview of imputation impact

In [160]:
incdecil_imp_results = df_all.groupby("COUNTRYW")[["INCDECIL", "INCDECIL_imputed"]].apply(utils.perc_missing)
incdecil_imp_results.to_csv(os.path.join(useful_paths.figure_dir, "03_eulfs", "eulfs_incdecile_per_country_after_imputing.csv"))
incdecil_imp_results

missing_obs_abs  missing_obs_rel
COUNTRYW                                                   
AT       INCDECIL                  86056.0         1.000000
         INCDECIL_imputed          86056.0         1.000000
BE       INCDECIL                   3061.0         0.157321
         INCDECIL_imputed            100.0         0.005140
CH       INCDECIL                   6786.0         0.172111
         INCDECIL_imputed            129.0         0.003272
CY       INCDECIL                   3254.0         0.183128
         INCDECIL_imputed            197.0         0.011087
CZ       INCDECIL                  17143.0         1.000000
         INCDECIL_imputed          17143.0         1.000000
DE       INCDECIL                  29833.0         0.113015
         INCDECIL_imputed             52.0         0.000197
DK       INCDECIL                   9054.0         0.185404
         INCDECIL_imputed            108.0         0.002212
EE       INCDECIL                   1628.0         0.113449
         INCDECIL_imputed             64.0         0.004460
ES       INCDECIL                  40695.0         1.000000
         INCDECIL_imputed          40695.0         1.000000
FI       INCDECIL                   8962.0         0.803479
         INCDECIL_imputed           1039.0         0.093150
FR       INCDECIL                   3944.0         0.128935
         INCDECIL_imputed            193.0         0.006309
GR       INCDECIL                  33797.0         0.456180
         INCDECIL_imputed            455.0         0.006141
HR       INCDECIL                   1097.0         0.341001
         INCDECIL_imputed            152.0         0.047249
HU       INCDECIL                   8746.0         0.106021
         INCDECIL_imputed            124.0         0.001503
IE       INCDECIL                  40103.0         0.635094
         INCDECIL_imputed            781.0         0.012368
IS       INCDECIL                   9087.0         1.000000
         INCDECIL_imputed           9087.0         1.000000
IT       INCDECIL                  46869.0         0.236949
         INCDECIL_imputed            221.0         0.001117
LT       INCDECIL                   6076.0         0.207273
         INCDECIL_imputed            102.0         0.003480
LU       INCDECIL                    772.0         0.172823
         INCDECIL_imputed             69.0         0.015447
LV       INCDECIL                    518.0         0.132549
         INCDECIL_imputed             72.0         0.018424
NL       INCDECIL                   6347.0         0.161616
         INCDECIL_imputed            143.0         0.003641
NO       INCDECIL                  14135.0         1.000000
         INCDECIL_imputed          14135.0         1.000000
PT       INCDECIL                  18188.0         0.283505
         INCDECIL_imputed           1353.0         0.021090
RO       INCDECIL                  25512.0         0.263103
         INCDECIL_imputed           5185.0         0.053472
SE       INCDECIL                  67463.0         1.000000
         INCDECIL_imputed          67463.0         1.000000
SK       INCDECIL                  12902.0         0.367045
         INCDECIL_imputed            375.0         0.010668
UK       INCDECIL                  14063.0         0.368276
         INCDECIL_imputed            230.0         0.006023

Save file

In [161]:
utils.save_df_to_files(df_all, output_dir=eulfs.path_eulfs_interim, fname_no_ext="eu_lfs_merged_{year}_incdecil_imputed".format(year=year))

#### Data preprocessing

Countries with data limitations, occupations:
•	ISCO-08 2D: BG, PL
•	ISCO-08 1D: MT

Countries with data limitations, regions:
•	NUTS 1: UK
•	NUTS Suppressed: NL

Countries with data limitations, income:
- AT, ES, SE, NO, CZ, IS

Countries removed for ISCO-08 3D results:
- BG
- PL
- MT

Reasons for dropping observations:
- ISCO not coded at 3D (but at 2 or 1D)
- COEFF is NAN
- ISCO3D is 633 (subsistence farming --> investigate: https://esco.ec.europa.eu/en/classification/occupation?uri=http://data.europa.eu/esco/isco/C633)

##### Attach covariates
- NACE labels
- GBN shares
- earnings deciles

Attach NACE labels and GBN shars

In [162]:
eulfs.join_covariates(
    year=2019,
    optional_input_dir=config["paths"]["interim"],
    input_fname_lfs="eu_lfs_merged_{year}_incdecil_imputed",
    covariates_by_isco=gbn_shares_no_wt,
    covariates_by_nace=covariates_by_nace,
    optional_output_dir=config["paths"]["interim"],
    output_fname_lfs="eu_lfs_merged_{year}_with_final_unweighted_shares_incdecil_imputed",
    isco_covariate_selection=None,
)

,AGE,COEFF,COEFF_share_brown_sl,COEFF_share_brown_slt,COEFF_share_green,COEFF_share_neutral_sl,COEFF_share_neutral_slt,COUNTRY,COUNTRYW,DEGURBA,...,SEX,WSTATOR,category_sl,category_slt,isco_code,share_brown_sl,share_brown_slt,share_green,share_neutral_sl,share_neutral_slt
0,52,82.5525,2.798390,1.399195,0.0000,79.754110,81.153305,AT,AT,3,...,1,1,neutral,neutral,132,0.033898,0.016949,0.00,0.966102,0.983051
1,57,39.1925,1.328559,0.664280,0.0000,37.863941,38.528220,AT,AT,1,...,1,1,neutral,neutral,132,0.033898,0.016949,0.00,0.966102,0.983051
2,37,76.8850,2.606271,1.303136,0.0000,74.278729,75.581864,AT,AT,3,...,1,1,neutral,neutral,132,0.033898,0.016949,0.00,0.966102,0.983051
3,47,70.1975,2.379576,1.189788,0.0000,67.817924,69.007712,AT,AT,3,...,1,1,neutral,neutral,132,0.033898,0.016949,0.00,0.966102,0.983051
4,37,25.4575,0.862966,0.431483,0.0000,24.594534,25.026017,AT,AT,2,...,1,1,neutral,neutral,132,0.033898,0.016949,0.00,0.966102,0.983051
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1412291,52,272.6200,0.000000,0.000000,0.0000,272.620000,272.620000,UK,UK,2,...,1,1,neutral,neutral,751,0.000000,0.000000,0.00,1.000000,1.000000
1412292,37,264.0200,0.000000,0.000000,0.0000,264.020000,264.020000,UK,UK,1,...,2,1,neutral,neutral,515,0.000000,0.000000,0.00,1.000000,1.000000
1412293,52,726.8300,670.920000,363.415000,0.0000,55.910000,363.415000,UK,UK,3,...,1,1,brown,brown,811,0.923077,0.500000,0.00,0.076923,0.500000
1412294,37,664.3200,0.000000,0.000000,0.0000,664.320000,664.320000,UK,UK,2,...,2,1,neutral,neutral,231,0.000000,0.000000,0.00,1.000000,1.000000


Match country-specific earnings distributions based on deciles

In [170]:
eulfs.join_covariates(
    year=2019,
    optional_input_dir=config["paths"]["interim"],
    input_fname_lfs="eu_lfs_merged_{year}_with_final_unweighted_shares_incdecil_imputed",
    covariates_by_isco=earnings_deciles,
    isco_join_col_eulfs=["COUNTRYW", "INCDECIL_imputed"],
    isco_join_col_covariates=["country", "decile"],
    optional_output_dir=config["paths"]["interim"],
    output_fname_lfs="eu_lfs_merged_{year}_with_final_unweighted_shares_and_earnings_incdecil_imputed",
    isco_covariate_selection=None,
)

,AGE,COEFF,COEFF_share_brown_sl,COEFF_share_brown_slt,COEFF_share_green,COEFF_share_neutral_sl,COEFF_share_neutral_slt,COUNTRY,COUNTRYW,DEGURBA,...,category_slt,country,decile,isco_code,share_brown_sl,share_brown_slt,share_green,share_neutral_sl,share_neutral_slt,year
0,52,82.5525,2.798390,1.399195,0.0000,79.754110,81.153305,AT,AT,3,...,neutral,NaN,NaN,132,0.033898,0.016949,0.00,0.966102,0.983051,NaN
1,57,39.1925,1.328559,0.664280,0.0000,37.863941,38.528220,AT,AT,1,...,neutral,NaN,NaN,132,0.033898,0.016949,0.00,0.966102,0.983051,NaN
2,37,76.8850,2.606271,1.303136,0.0000,74.278729,75.581864,AT,AT,3,...,neutral,NaN,NaN,132,0.033898,0.016949,0.00,0.966102,0.983051,NaN
3,47,70.1975,2.379576,1.189788,0.0000,67.817924,69.007712,AT,AT,3,...,neutral,NaN,NaN,132,0.033898,0.016949,0.00,0.966102,0.983051,NaN
4,37,25.4575,0.862966,0.431483,0.0000,24.594534,25.026017,AT,AT,2,...,neutral,NaN,NaN,132,0.033898,0.016949,0.00,0.966102,0.983051,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1412291,52,272.6200,0.000000,0.000000,0.0000,272.620000,272.620000,UK,UK,2,...,neutral,NaN,NaN,751,0.000000,0.000000,0.00,1.000000,1.000000,NaN
1412292,37,264.0200,0.000000,0.000000,0.0000,264.020000,264.020000,UK,UK,1,...,neutral,NaN,NaN,515,0.000000,0.000000,0.00,1.000000,1.000000,NaN
1412293,52,726.8300,670.920000,363.415000,0.0000,55.910000,363.415000,UK,UK,3,...,brown,NaN,NaN,811,0.923077,0.500000,0.00,0.076923,0.500000,NaN
1412294,37,664.3200,0.000000,0.000000,0.0000,664.320000,664.320000,UK,UK,2,...,neutral,NaN,NaN,231,0.000000,0.000000,0.00,1.000000,1.000000,NaN


##### Aggregate by regions & industries

In [164]:
# aggregate
agg_dict = {
    "COEFF": np.sum,
    "NOBS": np.sum,
    "COEFF_share_green": np.sum,
    "COEFF_share_brown_sl": np.sum,
    "COEFF_share_brown_slt": np.sum,
}

if reprocess:
    # by 1-digit industry
    eulfs.aggregate(
        year=2019,
        group_by=["NACE1D_label"],
        agg_dict=agg_dict,
        input_fname="eu_lfs_merged_{year}_with_final_unweighted_shares",
        output_fname="eulfs_{year}_by_{by}_final_unweighted_shares",
    )

    # by 1-digit industry and country
    eulfs.aggregate(
        year=2019,
        group_by=["NACE1D_label", "COUNTRYW"],
        agg_dict=agg_dict,
        input_fname="eu_lfs_merged_{year}_with_final_unweighted_shares",
        output_fname="eulfs_{year}_by_{by}_final_unweighted_shares",
    )

    # by nuts-2 regions and 1-digit industries
    eulfs.aggregate(
        year=2019,
        group_by=["NUTS_ID", "NACE1D_label"],
        agg_dict=agg_dict,
        input_fname="eu_lfs_merged_{year}_with_final_unweighted_shares",
        output_fname="eulfs_{year}_by_{by}_final_unweighted_shares",
    )